# Собираем исходник с Яндекс Диска

In [1]:
import yadisk
import pandas as pd
from io import BytesIO
import os 
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv("YANDEX_TOKEN")
DISK_PATH = os.getenv("DISK_PATH")

print(f"Токен загружен: {bool(TOKEN)}")

# Настройки отображения pandas, чтобы датафрейм выводился целиком
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

Токен загружен: True


In [2]:
def read_xlsx_from_personal_disk(token, disk_path, sheet_name=0):
    """
    Скачивает Excel с Яндекс.Диска и читает указанный лист.
    sheet_name может быть строкой (название листа), числом (индекс) или None (все листы).
    """
    y = yadisk.YaDisk(token=token)
    
    if not y.check_token():
        raise Exception("Неверный OAuth-токен")
        
    if not y.exists(disk_path):
        raise Exception(f"Файл не найден по пути: {disk_path}")

    try:
        # Создаем буфер в памяти и скачиваем туда файл
        buffer = BytesIO()
        y.download(disk_path, buffer)
        buffer.seek(0)
        
        # Читаем нужный лист с помощью движка 'calamine'
        df = pd.read_excel(buffer, engine='calamine', sheet_name=sheet_name)
        return df
        
    except Exception as e:
        raise Exception(f"Ошибка при загрузке или чтении Excel: {e}")

In [3]:
# TARGET_SHEET = "Закупки"

TARGET_SHEET = "ОС Главная"

try:
    print(f"Загрузка листа: '{TARGET_SHEET}'...")
    data_frame = read_xlsx_from_personal_disk(TOKEN, DISK_PATH, sheet_name=TARGET_SHEET)
    
    print(f"Успешно прочитано строк: {len(data_frame)}\n")
    print(data_frame)
    
except Exception as error:
    print(f"Произошла ошибка: {error}")

Загрузка листа: 'ОС Главная'...
Успешно прочитано строк: 20

       №         Код  Код ОКПД2               Номенклатура      QR код  Статус  Проект  МТО/Услуга  Цена  Сумма  Упаковка Ед. изм.  Кол-во  Строка плана закупки  План потребности  План МТР  Отдел МТО Куратор МТО  Счет от поставщика\n   согласован  Цена финальная  Сумма финальная  Заявка на закупку  Заявка на оплату  Приходный ордер  Оплачено Получено со склада  Ссылка  Остаток  Место хранения
0    1.0  А000000000        NaN                ТабуретОЧКА  А000000000     NaN     NaN         NaN   NaN    NaN       NaN      шт.       1                   NaN               NaN       NaN        NaN        Биба                                NaN            1000              NaN                NaN               NaN              NaN       NaN      Получено 2028     NaN        1              10
1    2.0  А000000001        NaN             Зеленый слоник  А000000001     NaN     NaN         NaN   NaN    NaN       NaN      шт.       1         

In [ ]:
# # Оба листа в виде словаря

# dfs = read_xlsx_from_personal_disk(TOKEN, DISK_PATH, sheet_name=["Закупки", "ОС Главная"])
# # Использование:
# print(dfs["Закупки"])
# print(dfs["ОС Главная"])

# Пересобираем эксельку на основании исходника (для полноты и разнообразия данных)

## На основе заданной структуры

In [9]:
import random
import numpy as np
from datetime import datetime, timedelta
import pandas as pd

# Фиксируем seed для воспроизводимости (опционально)
random.seed(42)

COUNT = 1000

In [5]:
# 1. Генерация уникальных 7-значных кодов
codes = random.sample(range(1000000, 9999999), COUNT)

# 2. Генерация уникальных кодов ОКПД2
okpd_list = [f"00.00.00.{i:04d}" for i in range(1, COUNT + 1)]

# Выборки для генерации номенклатуры
categories = [
    # Электроника и компоненты
    "Транзистор IRF1404", "Резистор 10 кОм (упаковка 100 шт)", "Светодиод 5мм (набор)", 
    "Конденсатор электролитический 1000мкФ", "Плата Arduino Nano V3.0", "Модуль ESP32 Wi-Fi/Bluetooth", 
    "Полетный контроллер Pixhawk 2.4.8", "Датчик давления воды MS5837", "Бесщеточный мотор A2212 1000KV", 
    "Регулятор оборотов ESC 20A BLHeli", "BMS плата защиты 3S 40A", "Преобразователь напряжения XL6009 DC-DC",
    "Кабель монтажный МГТФ 0.35 мм²", "Разъемы XT60 (пара «папа-мама»)", "Комплект Dupont коннекторов",

    # 3D-печать и расходники
    "Пластик PLA 1.75 мм (Черный, 1 кг)", "Пластик PETG 1.75 мм (Синий, 1 кг)", "Пластик ABS 1.75 мм (Серый, 1 кг)",
    "Сопло для 3D принтера E3D V6 0.4 мм", "Термобарьер для экструдера", "Шаговый двигатель NEMA 17", 
    "Стекло для стола 3D принтера 235x235", "Лента малярная термостойкая 50мм", "Ремень зубчатый GT2 (пог. метр)",

    # Инструменты и паяльное оборудование
    "Паяльник 120 Вт с регулировкой", "Набор сменных жал T12 (5 шт)", "Припой ПОС-61 с канифолью (100г)", 
    "Флюс паяльный ЛТИ-120 (флакон)", "Спирто-канифольный флюс (СКФ)", "Набор прецизионных отверток (24 в 1)", 
    "Кусачки бокорезы мини", "Штангенциркуль цифровой 150мм", "Мультиметр цифровой с прозвонкой", 
    "Осциллограф цифровой портативный", "Мини-паяльник USB", "Губка латунная для очистки жал",

    # Крепеж, металлопрокат и материалы
    "Винт M3x10 с цилиндрической головкой (100 шт)", "Гайка M3 самоконтрящаяся (100 шт)", "Шайба гровер M3 (100 шт)", 
    "Профиль алюминиевый конструкционный 2020 (1 м)", "Трубка термоусадочная 3мм (упаковка 5м)", 
    "Стяжки нейлоновые кабельные 100х2.5 мм (упак.)", "Лист стеклотекстолита фольгированного", 
    "Диск отрезной абразивный по металлу", "Пилка по металлу для лобзика (набор)",

    # Канцелярия и общие хозтовары
    "Маркеры для магнитной доски (набор 4 цвета)", "Маркеры перманентные черные (упаковка 10 шт)", 
    "Канцелярский нож усиленный с лезвиями", "Бумага для принтера А4 (пачка 500 листов)", 
    "Клей-карандаш канцелярский", "Скотч упаковочный прозрачный", "Блокнот для записей в клетку",
    "Очки защитные прозрачные", "Перчатки рабочие с полиуретановым покрытием (пара)"
]

statuses = ["Получено со склада", "Согласовано", "В доставке", "Ожидает оплаты", "Отменено"]
projects = ["ООО БибаИБоба", "Проект Робототехника", "НИОКР-2026", "Модернизация-МТО", "Цех №4"]
mto_types = ["МТО", "Услуга"]
units = ["шт.", "л.", "кг", "компл.", "упак.", "м"]
departments = ["101", "105", "111", "204", "302"]
curators = ["Иванов И.И.", "Петров П.П.", "Сидоров С.С.", "Смирнова А.В.", "Кузнецов М.Ю."]

data = []
start_date = datetime(2026, 1, 1)

for i in range(COUNT):
    code = codes[i]
    okpd = okpd_list[i]
    item_name = random.choice(categories)
    status = random.choice(statuses)
    project = random.choice(projects)
    mto = random.choice(mto_types)
    
    # Экономические показатели
    price = round(random.uniform(100, 50000), 2)
    quantity = random.randint(1, 100)
    total_price = round(price * quantity, 2)
    final_price = round(total_price * random.uniform(1.02, 1.15), 2)  # с учетом наценок/логистики
    unit = random.choice(units)
    
    # Даты и статус согласования
    random_days = random.randint(1, 250)
    base_date = start_date + timedelta(days=random_days)
    date_str = base_date.strftime("%d.%m.%Y")
    
    approved_plan = f"утверждена от {date_str}" if random.random() > 0.1 else "не утверждена"
    invoice_approved = random.choice(["Да", "Нет"])
    dept = random.choice(departments)
    curator = random.choice(curators)
    
    payment_request = f"АБ00-{random.randint(100000, 999999)} от {date_str}"
    
    # Связная логика статусов, дат и мест хранения
    paid_date = ""
    received_date = ""
    canceled_date = ""
    storage = ""
    
    if status == "Отменено":
        canceled_date = (base_date + timedelta(days=random.randint(1, 5))).strftime("%d.%m.%Y")
        storage = ""  # Место хранения пустое при отмене
    elif status == "Получено со склада":
        pay_d = base_date + timedelta(days=random.randint(1, 3))
        rec_d = pay_d + timedelta(days=random.randint(1, 10))
        paid_date = pay_d.strftime("%d.%m.%Y")
        received_date = f"Получено {rec_d.strftime('%d.%m.%Y')}"
        storage = f"{random.randint(10, 99)}В{random.randint(0, 9)}"
    elif status in ["Согласовано", "В доставке"]:
        if random.random() > 0.3:
            paid_date = (base_date + timedelta(days=random.randint(1, 5))).strftime("%d.%m.%Y")
    
    data.append({
        "Код": code,
        "Код ОКПД2": okpd,
        "Номенклатура": item_name,
        "Статус": status,
        "Проект": project,
        "МТО/Услуга": mto,
        "Цена": price,
        "Кол-во": quantity,
        "Сумма": total_price,
        "Ед. изм.": unit,
        "Строка плана закупки": approved_plan,
        "Счет поставщика согласован": invoice_approved,
        "Отдел МТО": dept,
        "Куратор МТО": curator,
        "Финальная сумма": final_price,
        "Заявка на оплату": payment_request,
        "Оплачено": paid_date,
        "Получено со склада": received_date,
        "Отменено": canceled_date,
        "Место хранения": storage
    })

In [6]:
df = pd.DataFrame(data)
output_filename = "synthetic_procurement_1000.xlsx"
df.to_excel(output_filename, index=False, engine='openpyxl')

print(f"Файл '{output_filename}' успешно сгенерирован! Записей: {len(df)}")

Файл 'synthetic_procurement_1000.xlsx' успешно сгенерирован! Записей: 1000


## Используя только исходный датасет на 20-30 записей

In [10]:
def generate_synthetic_from_source(source_df, target_count=1000):
    """
    Генерирует датасет на основе исходного DataFrame.
    """
    print(f"Анализ исходного файла: {len(source_df)} строк, {len(source_df.columns)} колонок.")
    
    # 1. Случайная выборка строк из исходника с повторениями (раздуваем до 1000)
    # ignore_index=True сбрасывает старые индексы
    synthetic_df = source_df.sample(n=target_count, replace=True, random_state=42).reset_index(drop=True)
    
    # 2. Делаем уникальные идентификаторы (Код и Код ОКПД2)
    # Предполагаем, что Код - это число (например, 1222222)
    start_code = 1000000
    synthetic_df['Код'] = range(start_code, start_code + target_count)
    synthetic_df['Код ОКПД2'] = [f"00.00.00.{str(i).zfill(4)}" for i in range(1, target_count + 1)]
    
    # 3. Мутируем числовые значения, чтобы данные не были идентичными клонами
    # Если колонки с ценой и количеством существуют, меняем их на случайную величину от -20% до +20%
    if 'Цена' in synthetic_df.columns and 'Кол-во' in synthetic_df.columns:
        # Изменяем количество (от 1 до 50 шт)
        synthetic_df['Кол-во'] = np.random.randint(1, 50, size=target_count)
        
        # Слегка меняем цену (± 20% от исходной)
        price_multipliers = np.random.uniform(0.8, 1.2, size=target_count)
        synthetic_df['Цена'] = (synthetic_df['Цена'] * price_multipliers).round(2)
        
        # Пересчитываем зависимые колонки
        if 'Сумма' in synthetic_df.columns:
            synthetic_df['Сумма'] = (synthetic_df['Цена'] * synthetic_df['Кол-во']).round(2)
            
        if 'Финальная сумма' in synthetic_df.columns:
            # Допустим, финальная сумма это Сумма + наценка (например, случайные 2-15%)
            markup = np.random.uniform(1.02, 1.15, size=target_count)
            synthetic_df['Финальная сумма'] = (synthetic_df['Сумма'] * markup).round(2)

    # 4. Мутируем заявки на оплату, чтобы они тоже выглядели уникальными
    if 'Заявка на оплату' in synthetic_df.columns:
        # Если формат "АБ00-123456 от 14.09.2026", просто меняем цифры до "от"
        def randomize_invoice(val):
            if pd.isna(val):
                return val
            val_str = str(val)
            if " от " in val_str:
                parts = val_str.split(" от ")
                return f"АБ00-{random.randint(100000, 999999)} от {parts[1]}"
            return val
            
        synthetic_df['Заявка на оплату'] = synthetic_df['Заявка на оплату'].apply(randomize_invoice)

    return synthetic_df

In [11]:
try:
    # 1. Читаем исходник (из Я.Диска)
    data_frame = read_xlsx_from_personal_disk(TOKEN, DISK_PATH, sheet_name="Закупки")
    
    # 2. Генерируем 1000 строк
    large_df = generate_synthetic_from_source(data_frame, target_count=1000)
    
    # 3. Сохраняем результат в новый файл на вашем компьютере
    output_filename = "synthetic_from_source.xlsx"
    large_df.to_excel(output_filename, index=False, engine='openpyxl')
    
    print(f"\nУспех! Сгенерирован файл '{output_filename}' на {len(large_df)} строк.")
    print(large_df.head())
    
except Exception as error:
    print(f"Произошла ошибка: {error}")

Анализ исходного файла: 24 строк, 31 колонок.

Успех! Сгенерирован файл 'synthetic_from_source.xlsx' на 1000 строк.
      №      Код      Код ОКПД2       Номенклатура      QR код              Статус        Проект МТО/Услуга    Цена    Сумма  Упаковка Ед. изм.  Кол-во   Строка плана закупки  План потребности              План МТР  Отдел МТО Куратор МТО Счет от поставщика\n   согласован  Цена финальная  Сумма финальная   Заявка на закупку           Заявка на оплату  Приходный ордер   Оплачено   Получено со склада  Отменено  Ссылка  Остаток Место хранения  Списано
0   7.0  1000000  00.00.00.0001          3d сканер  МТО1222222  Получено со склада  ОООБибаИБоба        МТО  101.81  1730.77       NaN       шт      17  утвержена от 14.09.32               NaN  00-000 от 24.09.2032        111        Биба                               Нет             110              110  2004 от 14.09.2026  АБ00-770487 от 14.09.2032              NaN 2026-09-20  Получено 15.09.2032       NaN     NaN      NaN     